# 🔄 Actualizar Dashboard Eco Go — Completo
Corre el refresh de **todas** las secciones del dashboard interno.

| Sección | Fuente |
|---|---|
| Precios (IPC) | `IPC TODESCA.xlsx` · `Gráficos de dispersión.xlsx` · `Cuadro Mensual.xlsx` |
| EMAE (Actividad) | `Base EsAE.xlsx` |
| Empleo + Salarios | `Empleo_nuevo.xlsx` · `Salarios.xlsx` |
| Tipo de Cambio | `TCR bandas.xlsx` · `Copia de Blue.xlsx` |
| Mercados | API EcoGo Markets |

> **Nota:** RPM, EsAE, RIGI, Reservas e Internacional · Consensus son exclusivos del dashboard clientes — actualizarlos desde `Actualizar_Dashboard_Clientes`.


In [ ]:
import sys, os

DASHBOARD_DIR = r"C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard"
sys.path.insert(0, DASHBOARD_DIR)

# Importar funciones del refresh
import refresh as R

print(f"Dashboard: {DASHBOARD_DIR}")
print(f"Excel base: {R.BASE_EXCEL}")
print(f"Data dir:   {R.DATA_DIR}")

## 1 · Precios (IPC)


In [ ]:
status = R.Status()
d = R.extract_precios(status)
if d:
    sz = R.save_data('precios', d)
    print(f"✅ precios.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 2 · EMAE (Actividad)


In [ ]:
status = R.Status()
d = R.extract_emae_series(status)
if d:
    sz = R.save_emae_series(d)
    print(f"✅ emae_series.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 3 · Empleo

In [ ]:
status = R.Status()
d = R.extract_empleo(status)
if d:
    sz = R.save_data('empleo', d)
    print(f"✅ empleo.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 4 · Salarios

In [ ]:
status = R.Status()
d = R.extract_salarios(status)
if d:
    sz = R.save_data('salarios', d)
    print(f"✅ salarios.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 5 · Tipo de Cambio

In [ ]:
status = R.Status()
d = R.extract_tipo_cambio(status)
if d:
    sz = R.save_data('tipo-cambio', d)
    print(f"✅ tipo-cambio.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 6 · Reservas

In [ ]:
status = R.Status()
d = R.extract_reservas(status)
if d:
    sz = R.save_data('reservas', d)
    print(f"✅ reservas.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 8 · Internacional (Monitor Mundial)

In [ ]:
import re, json as _json
from datetime import datetime as _dt

MONITOR_JS = R.MONITOR_MUNDIAL_JS
JS_PATH    = os.path.join(R.DATA_DIR, 'internacional.js')
JSON_PATH  = os.path.join(R.DATA_DIR, 'internacional.json')

if not os.path.exists(MONITOR_JS):
    print(f'⚠️  No se encontró: {MONITOR_JS}')
else:
    with open(MONITOR_JS, 'r', encoding='utf-8') as _f:
        src = _f.read()
    m = re.search(r'window\.MONITOR_DATA\s*=\s*(\{.*\});?\s*$', src, re.DOTALL)
    if not m:
        print('❌ No se pudo parsear monitor-data.js')
    else:
        intl_data = _json.loads(m.group(1))
        _str = _json.dumps(intl_data, ensure_ascii=False, default=str)
        # JSON
        try:
            with open(JSON_PATH, 'w', encoding='utf-8') as _f:
                _f.write(_str)
        except OSError:
            pass
        # JS
        js = (f'// Datos Internacional - regenerado por refresh.py el '
              f'{_dt.now().strftime("%Y-%m-%d %H:%M")}\n'
              f'window.INTERNACIONAL_DATA = {_str};\n')
        with open(JS_PATH, 'w', encoding='utf-8') as _f:
            _f.write(js)
        sz = os.path.getsize(JS_PATH)
        n_paises = len(intl_data.get('countries', []))
        print(f'✅ internacional.js — {sz:,} bytes · {n_paises} países')


## 7 · Mercados (API)

In [ ]:
import json, urllib.request
from datetime import datetime

API_URL = "https://ecogomarkets.honorio-zabaleta.workers.dev/data/dashboard_payload.json"
JS_PATH = os.path.join(R.DATA_DIR, 'mercados.js')

print("Conectando a la API de EcoGo Markets...")
req = urllib.request.Request(API_URL, headers={"User-Agent": "EcoGo-Dashboard-Refresh/1.0"})
with urllib.request.urlopen(req, timeout=30) as resp:
    payload = json.loads(resp.read().decode('utf-8'))

subset = {
    k: payload.get(k, {} if k not in ['fixed_curve','cer_curve','hero_metrics'] else [])
    for k in ['meta','external_context','overview','fixed_curve','fixed_curve_history',
               'cer_curve','cer_curve_history','dollar_linked','hard_dollar',
               'hard_dollar_curve_history','hero_metrics']
}
ts = datetime.now().strftime('%Y-%m-%d %H:%M')
js = '// Mercados data - ' + ts + '\nwindow.MERCADOS_DATA = ' + json.dumps(subset, ensure_ascii=False, default=str) + ';\n'
with open(JS_PATH, 'w', encoding='utf-8') as f:
    f.write(js)
print(f"✅ mercados.js — {os.path.getsize(JS_PATH):,} bytes")

## ✅ Resumen final

In [ ]:
from datetime import datetime
import os

data_dir = R.DATA_DIR
archivos = ['precios.js','emae_series.js','empleo.js','salarios.js','tipo-cambio.js','reservas.js','internacional.js','mercados.js']
print(f"{'Archivo':<22} {'Tamaño':>12} {'Modificado'}")
print('-' * 55)
for f in archivos:
    p = os.path.join(data_dir, f)
    if os.path.exists(p):
        sz = os.path.getsize(p)
        mt = datetime.fromtimestamp(os.path.getmtime(p)).strftime('%d/%m %H:%M')
        print(f"{f:<22} {sz:>10,} b  {mt}")
    else:
        print(f"{f:<22} {'—':>12}")
print()
print('Listo. Recargá el dashboard en el navegador.')